#Random Label Machine Unlearning with Retain Loss with param grid


In [1]:
!pip install -q transformers peft trl datasets accelerate

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from utils2 import load_rwku_data, prepare_tokenized_dataset, evaluate_model, evaluate_neighbours

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/Users/user/Desktop/school/master's <3/semester III/nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [3]:
# downloanding the model
MODEL_ID = "Qwen/Qwen2.5-3B"

SUBJECT   = "Donald Trump"

print(f"Loading model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
)
model = model.to(DEVICE)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj", "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
print("Number of parameters for training:")
peft_model.print_trainable_parameters()

Loading model: Qwen/Qwen2.5-3B


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:00<00:00, 31570.58it/s]


Number of parameters for training:
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [4]:
person_train, questions_forget, keywords_forget, questions_retain, keywords_retain = load_rwku_data("Donald Trump")
tokenized_forget_dataset = prepare_tokenized_dataset(person_train, tokenizer)

Questions for forgetting test: 20
Questions for general knowledge test: 30
Texts for training (unlearning): 226
Data is ready


In [5]:
print("BASELINE: model knowledge BEFORE unlearning")

print("EFFICACY — direct questions about Donald Trump (should be HIGH)")
acc_forget_before = evaluate_model(peft_model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("NEIGHBOURS — questions about associated topics (should be HIGH)")
acc_retain_before = evaluate_neighbours(peft_model, tokenizer, questions_retain, keywords_retain, DEVICE)

BASELINE: model knowledge BEFORE unlearning
EFFICACY — direct questions about Donald Trump (should be HIGH)
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the answer is "the apprentice."'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hosted the reality television series ___.
Expected: 'the apprentice'
Model generated: 'the answer is "the apprentice."'
Result: PASSED

--------------

In [7]:
# Build retain dataset from RWKU training passages for subjects other than the target.
# These passages are used in the retain loss pass so the model keeps general knowledge
# intact while the random-label pass pushes it away from target-specific knowledge.
from datasets import load_dataset as _load_ds

_all_train = _load_ds("jinzhuoran/RWKU", 'train_original_passage', split='train')
retain_raw = _all_train.filter(lambda x: SUBJECT not in x['subject'])
retain_raw = retain_raw.select(range(min(300, len(retain_raw))))
tokenized_retain_dataset = prepare_tokenized_dataset(retain_raw, tokenizer)
print(f"Retain dataset: {len(tokenized_retain_dataset)} passages (subjects other than {SUBJECT})")

Data is ready
Retain dataset: 300 passages (subjects other than Donald Trump)


In [8]:
import csv, os
os.environ["TQDM_DISABLE"] = "1"

from torch.utils.data import DataLoader
from transformers import TrainerCallback

class PrintProgress(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs and "loss" in logs:
            print(f"  step {state.global_step}/{state.max_steps}  loss={logs['loss']:.4f}")

def _collate_retain(batch):
    return {
        'input_ids':      torch.stack([torch.tensor(b['input_ids'])      for b in batch]),
        'attention_mask': torch.stack([torch.tensor(b['attention_mask']) for b in batch]),
        'labels':         torch.stack([torch.tensor(b['labels'])         for b in batch]),
    }

class RandomLabelTrainer(Trainer):
    def __init__(self, *args, retain_dataset=None, beta=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        if retain_dataset is not None:
            self.retain_loader = DataLoader(
                retain_dataset, batch_size=1, shuffle=True, collate_fn=_collate_retain,
            )
            self._retain_iter = iter(self.retain_loader)
        else:
            self.retain_loader = None

    def _next_retain_batch(self):
        try:
            return next(self._retain_iter)
        except StopIteration:
            self._retain_iter = iter(self.retain_loader)
            return next(self._retain_iter)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs = {k: v.clone() for k, v in inputs.items()}
        device = inputs['input_ids'].device

        mask = inputs['labels'] != -100
        random_labels = torch.randint(
            0, model.config.vocab_size, inputs['labels'].shape, device=device
        )
        inputs['labels'] = torch.where(mask, random_labels, inputs['labels'])

        forget_out  = model(**inputs)
        forget_loss = forget_out.loss
        total_loss  = forget_loss

        if self.retain_loader is not None:
            saved_out = forget_out if return_outputs else None
            del forget_out
            if device.type == 'mps':
                torch.mps.empty_cache()

            retain_batch = {k: v.to(device) for k, v in self._next_retain_batch().items()}
            retain_out  = model(**retain_batch)
            total_loss  = forget_loss + self.beta * retain_out.loss
            del retain_out
            if device.type == 'mps':
                torch.mps.empty_cache()

            return (total_loss, saved_out) if return_outputs else total_loss

        return (total_loss, forget_out) if return_outputs else total_loss


In [9]:
GRID = [
    {"lr": 5e-5, "beta": 0.3},
    {"lr": 5e-5, "beta": 0.8},
    {"lr": 5e-5, "beta": 1.5},
    {"lr": 1e-4, "beta": 0.3},
    {"lr": 1e-4, "beta": 0.8},
    {"lr": 1e-4, "beta": 1.5},
    {"lr": 2e-4, "beta": 0.3},
    {"lr": 2e-4, "beta": 0.8},
    {"lr": 2e-4, "beta": 1.5},
]

In [10]:
# Reuse baseline scores from cell-6 (acc_forget_before, acc_retain_before)
csv_path = "./unlearning_grid_results_Qwen2.5-3B.csv"
with open(csv_path, "w", newline="") as f:
    csv.DictWriter(f, fieldnames=[
        "model", "subject", "lr", "beta",
        "efficacy_before", "efficacy_after",
        "neighbours_before", "neighbours_after",
    ]).writeheader()

for config in GRID:
    lr, beta = config["lr"], config["beta"]
    print(f"\n{'='*60}")
    print(f"Config: lr={lr}  beta={beta}")
    print(f"{'='*60}")

    fresh_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16).to(DEVICE)
    fresh_peft  = get_peft_model(fresh_model, LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    ))

    trainer = RandomLabelTrainer(
        model=fresh_peft,
        args=TrainingArguments(
            output_dir=f"./unlearning_lr{lr}_beta{beta}",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=2,
            learning_rate=lr,
            max_steps=100,
            logging_steps=1,
            optim="adamw_torch",
            report_to="none",
        ),
        train_dataset=tokenized_forget_dataset,
        retain_dataset=tokenized_retain_dataset,
        beta=beta,
        callbacks=[PrintProgress()],
    )

    print("Training...")
    trainer.train()

    print("Evaluating AFTER unlearning...")
    eff_after = evaluate_model(fresh_peft, tokenizer, questions_forget, keywords_forget, DEVICE)
    nbr_after = evaluate_neighbours(fresh_peft, tokenizer, questions_retain, keywords_retain, DEVICE)

    print(f"Efficacy:   {acc_forget_before:.1f}% -> {eff_after:.1f}%")
    print(f"Neighbours: {acc_retain_before:.1f}% -> {nbr_after:.1f}%")

    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "model", "subject", "lr", "beta",
            "efficacy_before", "efficacy_after",
            "neighbours_before", "neighbours_after",
        ])
        writer.writerow({
            "model": MODEL_ID, "subject": SUBJECT,
            "lr": lr, "beta": beta,
            "efficacy_before":   f"{acc_forget_before:.1f}",
            "efficacy_after":    f"{eff_after:.1f}",
            "neighbours_before": f"{acc_retain_before:.1f}",
            "neighbours_after":  f"{nbr_after:.1f}",
        })

    del fresh_model, fresh_peft, trainer
    torch.mps.empty_cache()

print(f"\nAll done. Results saved to {csv_path}")


Config: lr=5e-05  beta=0.3


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 18889.14it/s]


Training...


/Users/user/Desktop/school/master's <3/semester III/nlp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,47.281990
2,42.974113
3,46.938728
4,42.420723
5,44.538605
6,42.231941
7,41.786079
8,44.707870
9,43.444519
10,43.886250


...

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 6101.93it/s]


Training...


/Users/user/Desktop/school/master's <3/semester III/nlp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,59.564240
2,48.664978
3,51.769302
4,45.246803
5,47.010109
6,40.968933
7,39.799149
8,40.670662
9,39.737457
10,39.957535


  step 1/100  loss=59.5642
  step 2/100  loss=48.6650
  step 3/100  loss=51.7693
  step 4/100  loss=45.2468
  step 5/100  loss=47.0101
  step 6/100  loss=40.9689
  step 7/100  loss=39.7991
  step 8/100  loss=40.6707
  step 9/100  loss=39.7375
  step 10/100  loss=39.9575
  step 11/100  loss=37.0654
  step 12/100  loss=37.0667
  step 13/100  loss=36.5585
  step 14/100  loss=32.3804
  step 15/100  loss=35.2938
  step 16/100  loss=32.6803
  step 17/100  loss=31.1174
  step 18/100  loss=33.4637
  step 19/100  loss=30.8625
  step 20/100  loss=32.1283
  step 21/100  loss=30.3040
  step 22/100  loss=32.5667
  step 23/100  loss=32.1545
  step 24/100  loss=28.3828
  step 25/100  loss=32.3148
  step 26/100  loss=34.8498
  step 27/100  loss=30.3337
  step 28/100  loss=31.9272
  step 29/100  loss=33.8002
  step 30/100  loss=29.0778
  step 31/100  loss=32.3375
  step 32/100  loss=29.6387
  step 33/100  loss=31.6200
  step 34/100  loss=30.3071
  step 35/100  loss=30.2257
  step 36/100  loss=28.8690
 